<span id="확률과-분류-기준을-구분하기"></span>

# Distinguish probability from classification criteria

**Create your own copy to run, edit, and record results.**

- Name: **⟦Enter here⟧**
- Student ID: **⟦Enter here⟧**

[Self-study material](https://chorok-daddy.github.io/niil/en/courses/ai-applications/logistic-regression/index.html)

- Run Code Cells: ▶ or `Shift+Enter`, in order from the top.
- Edit a Text Cell: double-click → write `your response` → `Shift+Enter`.
- Response cell: enter your own code.
- After changing values: rerun that cell and the relevant cells below it.

<span id="csv의-입력-확인"></span>

## 1. Check CSV inputs

The first 8 CSV columns are input features, and the last is the 0/1 Label. x=(x_raw-mean)/scale performs Standardization by subtracting each feature's mean and dividing by its standard deviation. This aligns feature scales; reuse the same mean and scale for new inputs.

<span id="a.-기준-예제-먼저-읽고-실행"></span>

### A. Baseline example · Read and run first

In [ ]:
import sys, numpy as np, matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
torch.manual_seed(42)
torch.set_num_threads(2)
print('Python:', sys.version.split()[0], '| PyTorch:', torch.__version__)
from pathlib import Path
from urllib.request import urlretrieve
p=Path('data_logistic_regression.csv')
if not p.exists():
    urlretrieve('https://chorok-daddy.github.io/niil/assets/courses/ai-applications/logistic-regression/data_logistic_regression.csv',p)
a=np.loadtxt(p,delimiter=',',dtype=np.float32)
x_raw=torch.tensor(a[:,:8]); labels=torch.tensor(a[:,-1],dtype=torch.long)
classes=torch.unique(labels);print('data:',a.shape,'classes:',classes)
mean=x_raw.mean(0);scale=x_raw.std(0).clamp_min(1e-6)
x=(x_raw-mean)/scale
y=labels.float().reshape(-1,1)
print(x.shape,y.shape)



<span id="b.-직접-수행"></span>

### B. Hands-on activities

1.  Run the data-loading cell. Print the shape, dtype, and first five rows of `x_raw` and `x` separately. Label the outputs before and after preprocessing.
2.  In the same cell, print the shape, dtype, and first five values of `labels` and `y`. Find where Labels are converted to the form required by Loss.
3.  Explain why you must reuse the training data's `mean` and `scale` to predict new samples, relating this to the scale of each Feature fed to the model.

Your prediction may describe a shape or an increasing/decreasing trend rather than an exact number. Use the example above to write code; answer explanation-only items in the Text Cell below.

In [ ]:
# ✍ Write your own: add your code below this cell.
# You may add Code Cells if the code gets long.


<span id="c.-직접-기록-이-text-cell을-더블클릭해서-수정"></span>

### C. Your record · Double-click this Text Cell to edit

- Conditions checked or changed: **⟦Enter your response⟧**
- Prediction before execution: **⟦Enter your response⟧**
- Value checked in the actual output/plot: **⟦Enter your response⟧**
- Difference from prediction and explanation: **⟦Enter your response⟧**

<span id="기준-학습-실행"></span>

## 2. Run baseline training

BCEWithLogitsLoss computes Sigmoid and Binary Cross Entropy together. Do not apply Sigmoid again during training; use it only to display probabilities and make Threshold decisions. For comparisons, specify the epoch count, as in train_classifier(lr=…, epochs=1000).

<span id="a.-기준-예제-먼저-읽고-실행-1"></span>

### A. Baseline example · Read and run first

In [ ]:
def train_classifier(lr=0.1,epochs=2000):
    torch.manual_seed(42);m=nn.Linear(8,1);opt=torch.optim.SGD(m.parameters(),lr=lr)
    criterion=nn.BCEWithLogitsLoss();losses=[];accuracies=[]
    for _ in range(epochs):
        m.train();logits=m(x);loss=criterion(logits,y)
        opt.zero_grad();loss.backward();opt.step()
        with torch.no_grad():
            logits=m(x);prediction=(torch.sigmoid(logits)>=0.5).long().reshape(-1)
            losses.append(criterion(logits,y).item());accuracies.append((prediction==labels).float().mean().item())
    return m,losses,accuracies
model,losses,accuracies=train_classifier()
print('Training accuracy:',accuracies[-1])
fig,axes=plt.subplots(1,2,figsize=(10,3));axes[0].plot(losses);axes[1].plot(accuracies)
for ax,yl in zip(axes,['Training loss','Training accuracy']):ax.set(xlabel='Epoch',ylabel=yl);ax.grid()
plt.show()



<span id="b.-직접-수행-1"></span>

### B. Hands-on activities

1.  Call `train_classifier` with Learning Rates $\eta\in\{0.001,0.01,0.1\}$, each for 1000 epochs. Keep the Seed and model initialization inside the function.
2.  Plot the three Loss curves together and tabulate the final Training Loss and Training Accuracy for each Learning Rate.
3.  Explain differences in the rate of Loss decrease by pointing to sections of the curves.

Your prediction may describe a shape or an increasing/decreasing trend rather than an exact number. Use the example above to write code; answer explanation-only items in the Text Cell below.

In [ ]:
# ✍ Write your own: add your code below this cell.
# You may add Code Cells if the code gets long.


<span id="c.-직접-기록-이-text-cell을-더블클릭해서-수정-1"></span>

### C. Your record · Double-click this Text Cell to edit

- Conditions checked or changed: **⟦Enter your response⟧**
- Prediction before execution: **⟦Enter your response⟧**
- Value checked in the actual output/plot: **⟦Enter your response⟧**
- Difference from prediction and explanation: **⟦Enter your response⟧**

<span id="출력을-해석하기"></span>

## 3. Interpret the output

Probabilities express how plausible each Class is, while the final Class is the single selected answer. Print probabilities alongside the selection to inspect both.

<span id="a.-기준-예제-먼저-읽고-실행-2"></span>

### A. Baseline example · Read and run first

In [ ]:
with torch.no_grad():
    probability=torch.sigmoid(model(x)).reshape(-1)
for threshold in [0.3,0.5]:
    prediction=(probability>=threshold).long()
    print(threshold,(prediction==labels).float().mean().item())
print(torch.stack([probability[:5],(probability[:5]>=.5).float(),labels[:5].float()],1))



<span id="b.-직접-수행-2"></span>

### B. Hands-on activities

1.  Choose one model trained in the previous activity and compute Probability once. Record the chosen Learning Rate. Apply Thresholds $\theta\in\{0.3,0.5,0.7\}$ separately to the same Probability.
2.  Record Threshold, Accuracy, and Positive prediction count in a three-row table.
3.  Choose up to 3 samples whose decisions change with the Threshold. Tabulate the sample index, Probability, target, and decisions at the three Thresholds. If no samples change decisions, state that.

Your prediction may describe a shape or an increasing/decreasing trend rather than an exact number. Use the example above to write code; answer explanation-only items in the Text Cell below.

In [ ]:
# ✍ Write your own: add your code below this cell.
# You may add Code Cells if the code gets long.


<span id="c.-직접-기록-이-text-cell을-더블클릭해서-수정-2"></span>

### C. Your record · Double-click this Text Cell to edit

- Conditions checked or changed: **⟦Enter your response⟧**
- Prediction before execution: **⟦Enter your response⟧**
- Value checked in the actual output/plot: **⟦Enter your response⟧**
- Difference from prediction and explanation: **⟦Enter your response⟧**

<span id="반복-숙달-실행-전에-판단하기"></span>

## Practice through repetition · Reason before running

Summarize the three conditions compared in the preceding activities in this table. You do not need to repeat experiments already completed. If comparisons are insufficient, supplement them by changing just one condition.

| Changed condition | Predicted shape/value/result | Actual result | Cause of any error |
|----|----|----|----|
| ⟦Enter your response 1⟧ | ⟦Enter your response⟧ | ⟦Enter your response⟧ | ⟦Enter your response⟧ |
| ⟦Enter your response 2⟧ | ⟦Enter your response⟧ | ⟦Enter your response⟧ | ⟦Enter your response⟧ |
| ⟦Enter your response 3⟧ | ⟦Enter your response⟧ | ⟦Enter your response⟧ | ⟦Enter your response⟧ |

In [ ]:
# ✍ Code for repeated experiments


<span id="선택-실습-오류의-원인을-찾아-코드-고치기"></span>

## Optional exercise · Find the cause of an error and fix the code

**When the BCE Target shape differs**

1.  After training the model, add a Code Cell at the bottom of the notebook and run the following code.

    <span id="cb1"></span>

    ``` python
    with torch.no_grad():
        debug_logits = model(x)
    print("Logits:", debug_logits.shape)
    for debug_target in [y.squeeze(1), y]:
        print("Target:", debug_target.shape)
        try:
            print("Loss:", F.binary_cross_entropy_with_logits(
                debug_logits, debug_target).item())
        except (RuntimeError, ValueError) as error:
            print(type(error).__name__, str(error))
    ```

2.  In a Text Cell, tabulate the Logits shape, each Target shape, and execution success. Explain which axis `squeeze(1)` removed and how Logits and Target shapes must match for this Loss.

<span id="저장-전-확인"></span>

## Check before saving

1.  Check the response fields, outputs, and plots, then run all cells in order in a fresh runtime.
2.  Save to your Drive, then download using `.ipynb`.

Include code, actual outputs and plots, and your own interpretation. Submit through KLAS; check the semester schedule and latest announcements for the deadline.

Filename: `Name_StudentID_Week7.ipynb`.